# LangChain + CoordiNode: Graph Chain

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/structured-world/coordinode-python/blob/main/demo/notebooks/02_langchain_graph_chain.ipynb)

Demonstrates `CoordinodeGraph` as a Knowledge Graph backend for LangChain.

**What works right now:**
- `graph.query()` — arbitrary Cypher pass-through
- `graph.schema` / `refresh_schema()` — live graph schema
- `add_graph_documents()` — add Nodes + Relationships from a LangChain `GraphDocument`
- `GraphCypherQAChain` — LLM generates Cypher from a natural-language question *(requires `OPENAI_API_KEY`)*

**Environments:**
- **Google Colab** — uses `coordinode-embedded` (in-process Rust engine, no server needed). First run compiles from source (~5 min); subsequent runs use the pip cache.
- **Local / Docker Compose** — connects to a running CoordiNode server via gRPC.

## Install dependencies

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

# Install coordinode-embedded in Colab only (requires Rust build).
if IN_COLAB and not os.environ.get("COORDINODE_ADDR"):
    # Install Rust toolchain via rustup (https://rustup.rs).
    # Colab's apt packages ship rustc ≤1.75, which cannot build coordinode-embedded
    # (requires Rust ≥1.80 for maturin/pyo3). apt-get is not a viable alternative here.
    # Download the installer to a temp file and execute it explicitly — this avoids
    # piping remote content directly into a shell while maintaining HTTPS/TLS security
    # through Python's default ssl context (cert-verified, TLS 1.2+).
    # SHA256 pinning of rustup-init is intentionally omitted: rustup.rs does not
    # publish a stable per-release checksum for sh.rustup.rs itself (only for
    # platform-specific rustup-init binaries), and pinning a hash here would break
    # silently on every rustup release. The HTTPS/TLS verification + temp-file
    # execution (not piped to shell) is the rustup team's recommended trust model.
    # No additional env-var gate (e.g. COORDINODE_ENABLE_RUSTUP) is needed:
    # the `IN_COLAB` check above already ensures this block never runs outside
    # Colab sessions, so there is no risk of unintentional execution in local
    # or server environments.
    import ssl as _ssl, tempfile as _tmp, urllib.request as _ur

    _ctx = _ssl.create_default_context()
    with _tmp.NamedTemporaryFile(mode="wb", suffix=".sh", delete=False) as _f:
        with _ur.urlopen("https://sh.rustup.rs", context=_ctx, timeout=30) as _r:
            _f.write(_r.read())
        _rustup_path = _f.name
    try:
        subprocess.run(["/bin/sh", _rustup_path, "-s", "--", "-y", "-q"], check=True, timeout=300)
    finally:
        os.unlink(_rustup_path)
    # Add cargo to PATH so maturin/pip can find it.
    _cargo_bin = os.path.expanduser("~/.cargo/bin")
    os.environ["PATH"] = f"{_cargo_bin}{os.pathsep}{os.environ.get('PATH', '')}"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "maturin"], check=True, timeout=300)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/structured-world/coordinode-python.git@bd8d0dc#subdirectory=coordinode-embedded",
        ],
        check=True,
        timeout=600,
    )

# coordinode-embedded is pinned to a specific git commit because it requires a Rust
# build (maturin/pyo3) and the embedded engine must match the Python SDK version.
# The remaining packages (coordinode, LangChain, etc.) are installed without pins:
# they are pure Python, release frequently, and pip resolves a compatible version.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "coordinode",
        "langchain",
        "langchain-coordinode",
        "langchain-community",
        "langchain-openai",
    ],
    check=True,
    timeout=300,
)

print("SDK installed")

## Adapter for embedded mode

`LocalClient` (embedded engine) has the same `.cypher()` API as `CoordinodeClient`.
The `_EmbeddedAdapter` below adds the extra methods that `CoordinodeGraph` expects
when it receives a pre-built `client=` object.

In [ ]:
class _EmbeddedAdapter:
    """Thin wrapper around LocalClient that adds CoordinodeClient-compatible methods."""

    def __init__(self, local_client):
        self._lc = local_client

    def cypher(self, query, params=None):
        return self._lc.cypher(query, params or {})

    def get_schema_text(self):
        lbls = self._lc.cypher("MATCH (n) UNWIND labels(n) AS lbl RETURN DISTINCT lbl ORDER BY lbl")
        rels = self._lc.cypher("MATCH ()-[r]->() RETURN DISTINCT type(r) AS t ORDER BY t")
        lines = ["Node labels:"]
        for r in lbls:
            lines.append(f"  - {r['lbl']}")
        lines.append("\nEdge types:")
        for r in rels:
            lines.append(f"  - {r['t']}")
        return "\n".join(lines)

    # Vector search not available in embedded mode — requires running CoordiNode server.

    def close(self):
        self._lc.close()


## Connect to CoordiNode

In [ ]:
import os, socket


def _port_open(port):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True
    except OSError:
        return False


if os.environ.get("COORDINODE_ADDR"):
    COORDINODE_ADDR = os.environ["COORDINODE_ADDR"]
    from coordinode import CoordinodeClient

    client = CoordinodeClient(COORDINODE_ADDR)
    print(f"Connected to {COORDINODE_ADDR}")
else:
    try:
        grpc_port = int(os.environ.get("COORDINODE_PORT", "7080"))
    except ValueError as exc:
        raise RuntimeError("COORDINODE_PORT must be an integer") from exc

    if _port_open(grpc_port):
        COORDINODE_ADDR = f"localhost:{grpc_port}"
        from coordinode import CoordinodeClient

        client = CoordinodeClient(COORDINODE_ADDR)
        print(f"Connected to {COORDINODE_ADDR}")
    else:
        # No server available — use the embedded in-process engine.
        # Works without Docker or any external service; data is in-memory.
        try:
            from coordinode_embedded import LocalClient
        except ImportError as exc:
            raise RuntimeError(
                "coordinode-embedded is not installed. "
                "Run: pip install git+https://github.com/structured-world/coordinode-python.git@bd8d0dc#subdirectory=coordinode-embedded"
                " — or start a CoordiNode server and set COORDINODE_ADDR."
            ) from exc

        _lc = LocalClient(":memory:")
        client = _EmbeddedAdapter(_lc)
        print("Using embedded LocalClient (in-process)")

## Create the graph store

Pass the pre-built `client=` object so the store doesn't open a second connection.

In [ ]:
import os, uuid
from langchain_coordinode import CoordinodeGraph
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
from langchain_core.documents import Document

graph = CoordinodeGraph(client=client)
print("Connected. Schema preview:")
print(graph.schema[:300])

## 1. add_graph_documents

LangChain `GraphDocument` wraps nodes and relationships from an LLM extraction pipeline.
Here we build one manually to show the insert path.

In [ ]:
tag = uuid.uuid4().hex

nodes = [
    Node(id=f"Turing-{tag}", type="Scientist", properties={"born": 1912, "field": "computer science"}),
    Node(id=f"Shannon-{tag}", type="Scientist", properties={"born": 1916, "field": "information theory"}),
    Node(id=f"Cryptography-{tag}", type="Field", properties={"era": "modern"}),
]
rels = [
    Relationship(source=nodes[0], target=nodes[2], type="FOUNDED", properties={"year": 1936}),
    Relationship(source=nodes[1], target=nodes[2], type="CONTRIBUTED_TO"),
    Relationship(source=nodes[0], target=nodes[1], type="INFLUENCED"),
]
doc = GraphDocument(nodes=nodes, relationships=rels, source=Document(page_content="Turing and Shannon"))
graph.add_graph_documents([doc])
print("Documents added")

## 2. query — direct Cypher

In [ ]:
rows = graph.query(
    "MATCH (s:Scientist)-[r]->(f:Field)"
    " WHERE s.name STARTS WITH $prefix"
    " RETURN s.name AS scientist, type(r) AS relation, f.name AS field",
    params={"prefix": f"Turing-{tag}"},
)
print("Scientists → Fields:")
for r in rows:
    print(f"  {r['scientist']}  --[{r['relation']}]-->  {r['field']}")

## 3. refresh_schema — structured_schema dict

In [ ]:
graph.refresh_schema()
print("node_props keys:", list(graph.structured_schema.get("node_props", {}).keys())[:10])
print("relationships:", graph.structured_schema.get("relationships", [])[:5])

## 4. Idempotency check

`add_graph_documents` uses MERGE internally — adding the same document twice must not
create duplicate edges.

In [ ]:
graph.add_graph_documents([doc])  # second upsert — must not create a duplicate edge
cnt = graph.query(
    "MATCH (a {name: $src})-[r:FOUNDED]->(b {name: $dst}) RETURN count(r) AS cnt",
    params={"src": f"Turing-{tag}", "dst": f"Cryptography-{tag}"},
)
print("FOUNDED edge count after double upsert (expect 1):", cnt[0]["cnt"])

## 5. GraphCypherQAChain — LLM-powered Cypher (optional)

> **This section requires `OPENAI_API_KEY`.** Set it in your environment or via
> `os.environ['OPENAI_API_KEY'] = 'sk-...'` before running.
> The cell is skipped automatically when the key is absent.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print(
        'Skipping: OPENAI_API_KEY is not set. Set it via os.environ["OPENAI_API_KEY"] = "sk-..." and re-run this cell.'
    )
else:
    from langchain.chains import GraphCypherQAChain
    from langchain_openai import ChatOpenAI

    chain = GraphCypherQAChain.from_llm(
        ChatOpenAI(model="gpt-4o-mini", temperature=0),
        graph=graph,
        verbose=True,
        allow_dangerous_requests=True,
    )
    result = chain.invoke(f"Who influenced Shannon-{tag}?")
    print("Answer:", result["result"])

## Cleanup

In [ ]:
# DETACH DELETE atomically removes all edges then the node in one operation.
# Two-step MATCH (n)-[r]-() / DELETE r / DELETE n is avoided because an
# undirected MATCH returns each edge from both endpoints, so the second pass
# fails with "cannot delete node with connected edges".
graph.query("MATCH (n) WHERE n.name ENDS WITH $tag DETACH DELETE n", params={"tag": tag})
print("Cleaned up")
graph.close()
client.close()  # injected client — owned by caller